# Main Results

This notebook summarizes the final 8-city x 8-task benchmark evaluation.
It reads `results/main_eval/summary.csv` and writes paper tables and figures under `results/main_eval/`. The final AETHER rows are already stored under the released `aether` model id in the standard summary and manifest.


In [ ]:
def _metric_std_col(metric):
    candidates = [f'{metric}_std', 'primary_std']
    for c in candidates:
        if c in summary.columns:
            return c
    return None

def _plot_grouped_bars_with_std(
    ax,
    data,
    *,
    x_col,
    y_col,
    std_col=None,
    hue_col=None,
    x_order=None,
    hue_order=None,
    palette=None,
    group_width=0.78,
    capsize=1.8,
    lw=0.52,
    edgecolor='#fbfbfb',
    edge_lw=0.42,
):
    """Draw grouped bars with explicit x coordinates and precomputed std.

    This avoids inferring positions from seaborn containers, which can create
    empty patches for categorical hue/order combinations.
    """
    if data.empty:
        return [], []
    plot_data = data.copy()
    plot_data['_x_key'] = plot_data[x_col].astype(str)
    if x_order is None:
        x_order = plot_data['_x_key'].dropna().drop_duplicates().tolist()
    else:
        x_order = [str(v) for v in x_order]
    x_pos = np.arange(len(x_order), dtype=float)

    if palette is None:
        palette = {}

    handles = []
    labels = []
    if hue_col is not None:
        plot_data['_hue_key'] = plot_data[hue_col].astype(str)
        if hue_order is None:
            hue_order = plot_data['_hue_key'].dropna().drop_duplicates().tolist()
        else:
            hue_order = [str(v) for v in hue_order]
        n_hue = max(len(hue_order), 1)
        bar_width = group_width / n_hue
        offsets = (np.arange(n_hue, dtype=float) - (n_hue - 1) / 2.0) * bar_width
        lookup = plot_data.drop_duplicates(['_x_key', '_hue_key'], keep='last').set_index(['_x_key', '_hue_key'])
        for h_idx, hue_key in enumerate(hue_order):
            xs, ys, errs = [], [], []
            for x_idx, x_key in enumerate(x_order):
                key = (x_key, hue_key)
                if key not in lookup.index:
                    continue
                row = lookup.loc[key]
                y = row.get(y_col, np.nan)
                if not np.isfinite(y):
                    continue
                err = row.get(std_col, 0.0) if std_col in lookup.columns else 0.0
                xs.append(x_pos[x_idx] + offsets[h_idx])
                ys.append(float(y))
                errs.append(float(err) if np.isfinite(err) else 0.0)
            if not xs:
                continue
            bars = ax.bar(
                xs, ys,
                width=bar_width * 0.92,
                yerr=errs,
                label=hue_key,
                color=palette.get(hue_key, None),
                edgecolor=edgecolor,
                linewidth=edge_lw,
                error_kw={
                    'ecolor': '#4a4a4a',
                    'elinewidth': lw,
                    'capsize': capsize,
                    'capthick': lw,
                    'alpha': 0.78,
                },
                zorder=3,
            )
            handles.append(bars[0])
            labels.append(hue_key)
    else:
        bar_width = group_width
        lookup = plot_data.drop_duplicates(['_x_key'], keep='last').set_index('_x_key')
        for x_idx, x_key in enumerate(x_order):
            if x_key not in lookup.index:
                continue
            row = lookup.loc[x_key]
            y = row.get(y_col, np.nan)
            if not np.isfinite(y):
                continue
            err = row.get(std_col, 0.0) if std_col in lookup.columns else 0.0
            bar = ax.bar(
                [x_pos[x_idx]], [float(y)],
                width=bar_width,
                yerr=[float(err) if np.isfinite(err) else 0.0],
                label=x_key,
                color=palette.get(x_key, None),
                edgecolor=edgecolor,
                linewidth=edge_lw,
                error_kw={
                    'ecolor': '#4a4a4a',
                    'elinewidth': lw,
                    'capsize': capsize,
                    'capthick': lw,
                    'alpha': 0.78,
                },
                zorder=3,
            )
            handles.append(bar[0])
            labels.append(x_key)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_order)
    ax.grid(axis='y', zorder=0)
    return handles, labels



In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'results').exists() and (ROOT.parent / 'results').exists():
    ROOT = ROOT.parent.resolve()
RESULT_DIR = ROOT / 'results' / 'main_eval'
TABLE_DIR = RESULT_DIR / 'paper_tables'
DIAG_DIR = RESULT_DIR / 'diagnostics'
FIG_DIR = RESULT_DIR / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
DIAG_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = ['aether','alphaearth','tessera','satclip','muse','urban2vec','cityfm','calliper','space2vec','place2vec','sphere2vec_fixed']
MODEL_LABEL = {
    'aether':'AETHER',
    'alphaearth':'AlphaEarth',
    'tessera':'TESSERA',
    'calliper':'CaLLiPer',
    'place2vec':'Place2Vec',
    'space2vec':'Space2Vec',
    'sphere2vec_fixed':'PE',
    'satclip':'SatCLIP',
    'cityfm':'CityFM',
    'muse':'MuseCL',
    'urban2vec':'Urban2Vec',
}
CITY_ORDER = ['london','new_york','singapore','sydney','mumbai','nairobi','jakarta','cape_town']
TASK_ORDER = ['landuse','road_density','population','age_distribution','gdp','nightlight','pm25','lst_day_mean']
TASK_TITLE = {
    'landuse':'LUC',
    'road_density':'RDE',
    'population':'POP',
    'age_distribution':'AGE*',
    'gdp':'GDP',
    'nightlight':'NTL',
    'pm25':'PM25',
    'lst_day_mean':'LST',
}
TASK_METRICS = {
    'population':['R2','MAE','RMSE'],
    'gdp':['R2','MAE','RMSE'],
    'nightlight':['R2','MAE','RMSE'],
    'pm25':['R2','MAE','RMSE'],
    'age_distribution':['KL','L1','Chebyshev'],
    'landuse':['F1_macro','precision_macro','recall_macro'],
    'road_density':['R2','MAE','RMSE'],
    'lst_day_mean':['R2','MAE','RMSE'],
}
PRIMARY_METRIC = {
    'population':'R2', 'gdp':'R2', 'nightlight':'R2', 'pm25':'R2',
    'age_distribution':'KL', 'landuse':'F1_macro', 'road_density':'R2', 'lst_day_mean':'R2',
}
LOWER_IS_BETTER = {'KL', 'MAE', 'RMSE', 'MSE', 'L1', 'Chebyshev'}

summary = pd.read_csv(RESULT_DIR / 'summary.csv')


manifest_path = ROOT / 'baselines' / 'registry' / 'embedding_manifest.csv'
if not manifest_path.exists():
    manifest_path = DIAG_DIR / 'manifest.csv' if (DIAG_DIR / 'manifest.csv').exists() else RESULT_DIR / 'manifest.csv'
manifest = pd.read_csv(manifest_path)
manifest.to_csv(DIAG_DIR / 'manifest.csv', index=False)
summary['model_label'] = summary['model'].map(MODEL_LABEL).fillna(summary.get('model_label', summary['model']).astype(str))
front_cols = ['model', 'model_label', 'city', 'task_id', 'task', 'protocol', 'task_type', 'n_samples']
summary = summary[[c for c in front_cols if c in summary.columns] + [c for c in summary.columns if c not in front_cols]]
summary.to_csv(RESULT_DIR / 'summary.csv', index=False)
summary['model'] = pd.Categorical(summary['model'], MODEL_ORDER, ordered=True)
summary['city'] = pd.Categorical(summary['city'], CITY_ORDER, ordered=True)
summary['task'] = pd.Categorical(summary['task'], TASK_ORDER, ordered=True)
summary = summary.sort_values(['task','city','model'])
coverage = (
    summary.groupby(['model','task']).size()
    .unstack(fill_value=0)
    .reindex(index=MODEL_ORDER, columns=TASK_ORDER, fill_value=0)
)
coverage.reset_index().to_csv(DIAG_DIR / 'coverage_by_model_task.csv', index=False)
sns.set_theme(style='whitegrid', context='notebook')
MODEL_COLORS = {
    # Matplotlib tab10-style palette requested for dense grouped bars.
    'aether': '#1f77b4',
    'alphaearth': '#ff7f0e',
    'tessera': '#2ca02c',
    'cityfm': '#d62728',
    'calliper': '#9467bd',
    'space2vec': '#8c564b',
    'place2vec': '#e377c2',
    'urban2vec': '#7f7f7f',
    'muse': '#bcbd22',
    'satclip': '#17becf',
    'sphere2vec_fixed': '#5b3a29',
}
PALETTE = {MODEL_LABEL[m]: MODEL_COLORS[m] for m in MODEL_ORDER}

print('summary rows:', len(summary))
print('manifest rows:', len(manifest), 'available:', int(manifest['available'].astype(bool).sum()))
display(summary.head())


## Alignment Audit

In [ ]:
rows = []
for p in sorted(RESULT_DIR.glob('*/*/*/*/aligned/alignment_report.json')):
    model, city, task_id, protocol = p.relative_to(RESULT_DIR).parts[:4]
    task = task_id.split('.')[1]
    report = json.loads(p.read_text())
    rows.append({
        'model': model,
        'model_label': MODEL_LABEL.get(model, model),
        'city': city,
        'task': task,
        'task_id': task_id,
        'aligner': report.get('aligner'),
        'embedding_dim': report.get('embedding_dim'),
        'n_task_samples': report.get('n_task_samples'),
        'missing_rate': report.get('missing_rate'),
        'hit_rate': 1.0 - float(report.get('missing_rate', 0.0)),
        'embedding_path': report.get('embedding', {}).get('path'),
    })

if rows:
    alignment = pd.DataFrame(rows)
else:
    # The compact release stores aggregated summaries rather than every per-run
    # alignment report. In that case we export a current manifest-based audit so
    # task/model coverage and path policies remain checkable without reusing
    # superseded alignment summary files from an earlier run.
    sample_cols = ['model', 'city', 'task', 'task_id', 'n_samples']
    sample_counts = summary[sample_cols].drop_duplicates(['model', 'city', 'task']) if set(sample_cols).issubset(summary.columns) else pd.DataFrame()
    alignment = manifest.merge(
        sample_counts,
        on=['model', 'city', 'task', 'task_id'],
        how='left',
    )
    alignment['model_label'] = alignment['model'].map(MODEL_LABEL).fillna(alignment['model'])
    alignment['aligner'] = alignment.get('alignment_policy', pd.Series(index=alignment.index, dtype=object))
    alignment['embedding_dim'] = np.nan
    alignment['n_task_samples'] = alignment.get('n_samples', pd.Series(index=alignment.index, dtype=float))
    alignment['missing_rate'] = np.nan
    alignment['hit_rate'] = np.nan

alignment.to_csv(DIAG_DIR / 'alignment_audit.csv', index=False)
print('alignment audit rows:', len(alignment))
print('alignment audit tasks:', sorted(alignment['task'].dropna().astype(str).unique()))

audit_cols = ['model','model_label','city','task','task_id','aligner','embedding_dim','n_task_samples','missing_rate','hit_rate','embedding_path']
display(alignment[[c for c in audit_cols if c in alignment.columns]].head())

if alignment['hit_rate'].notna().any():
    model_hit = alignment.groupby(['model','model_label']).agg(
        runs=('task_id','count'),
        total_samples=('n_task_samples','sum'),
        mean_hit_rate=('hit_rate','mean'),
        min_hit_rate=('hit_rate','min'),
        max_missing_rate=('missing_rate','max'),
    ).reset_index()
    missing_samples = (alignment['missing_rate'].astype(float) * alignment['n_task_samples'].astype(float)).round()
    weighted = alignment.assign(missing_samples=missing_samples).groupby(['model','model_label']).agg(
        missing_samples=('missing_samples','sum'),
        total_samples=('n_task_samples','sum'),
    ).reset_index()
    weighted['weighted_hit_rate'] = 1.0 - weighted['missing_samples'] / weighted['total_samples']
    model_hit = model_hit.merge(weighted[['model','model_label','missing_samples','weighted_hit_rate']], on=['model','model_label'])
    model_hit['model'] = pd.Categorical(model_hit['model'], MODEL_ORDER, ordered=True)
    model_hit = model_hit.sort_values('model')
    model_hit.to_csv(DIAG_DIR / 'model_alignment_hit_rate_summary.csv', index=False)
    display(model_hit[['model_label','runs','weighted_hit_rate','mean_hit_rate','min_hit_rate','max_missing_rate','missing_samples']])

    fig, ax = plt.subplots(figsize=(12, 4.2))
    sns.barplot(
        data=model_hit, x='model_label', y='weighted_hit_rate',
        hue='model_label', palette=PALETTE, legend=False, ax=ax,
    )
    ax.set_ylim(0, 1.02)
    ax.set_title('Weighted embedding hit rate by model')
    ax.set_xlabel('')
    ax.set_ylabel('Weighted hit rate')
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    fig.tight_layout()
    out = FIG_DIR / 'weighted_alignment_hit_rate_by_model.png'
    fig.savefig(out, dpi=240, bbox_inches='tight')
    print(out)
else:
    superseded = FIG_DIR / 'weighted_alignment_hit_rate_by_model.png'
    if superseded.exists():
        superseded.unlink()
    superseded_summary = DIAG_DIR / 'model_alignment_hit_rate_summary.csv'
    if superseded_summary.exists():
        superseded_summary.unlink()
    print('Compact release has no per-run alignment reports; skipped hit-rate plot.')

population_mentions = manifest[
    manifest['task'].isin(['nightlight','landuse'])
    & manifest[['embedding_path','source_role','alignment_policy']].astype(str).apply(
        lambda s: s.str.contains('population', case=False, na=False)
    ).any(axis=1)
]
print('NTL/landuse manifest rows mentioning population:', len(population_mentions))
display(population_mentions[['model','city','task','embedding_path','source_role','alignment_policy']])


## Primary Metric By Model And Task

In [ ]:
TASK_EVAL_CITIES = {task: CITY_ORDER for task in TASK_ORDER}
TASK_EVAL_CITIES['age_distribution'] = ['london', 'new_york', 'singapore', 'sydney']

city_rank_rows = []
primary_rows = []
for task in TASK_ORDER:
    metric = PRIMARY_METRIC[task]
    lower = metric in LOWER_IS_BETTER
    task_cities = TASK_EVAL_CITIES.get(task, CITY_ORDER)
    sub = summary[
        summary['task'].astype(str).eq(task)
        & summary['city'].astype(str).isin(task_cities)
    ].copy()
    sub[metric] = pd.to_numeric(sub[metric], errors='coerce')
    sub = sub.dropna(subset=[metric])

    task_city_rank_rows = []
    for city in task_cities:
        csub = sub[sub['city'].astype(str).eq(city)].copy()
        if csub.empty:
            continue
        csub['city_rank'] = csub[metric].rank(ascending=lower, method='min')
        for _, row in csub.iterrows():
            task_city_rank_rows.append({
                'task': task,
                'city': city,
                'metric': metric,
                'higher_is_better': not lower,
                'model': str(row['model']),
                'model_label': row['model_label'],
                'score': float(row[metric]),
                'city_rank': float(row['city_rank']),
            })
    city_rank_rows.extend(task_city_rank_rows)
    city_rank_df_task = pd.DataFrame(task_city_rank_rows)

    for model in MODEL_ORDER:
        vals = sub.loc[sub['model'].astype(str).eq(model), metric].dropna()
        ranks = city_rank_df_task.loc[city_rank_df_task['model'].eq(model), 'city_rank'].dropna()
        if vals.empty or ranks.empty:
            continue
        primary_rows.append({
            'task': task,
            'metric': metric,
            'higher_is_better': not lower,
            'model': model,
            'model_label': MODEL_LABEL[model],
            'n': int(len(vals)),
            'mean': float(vals.mean()),
            'std': float(vals.std(ddof=0)),
            'mean_city_rank': float(ranks.mean()),
            'city_rank_std': float(ranks.std(ddof=0)),
            'rank_n_cities': int(len(ranks)),
            'rank_method': 'mean_city_rank_after_seed_average',
        })

city_rank = pd.DataFrame(city_rank_rows)
city_rank.to_csv(TABLE_DIR / 'primary_metric_city_rank_long.csv', index=False)
primary = pd.DataFrame(primary_rows)
primary['rank'] = primary['mean_city_rank']
primary.to_csv(TABLE_DIR / 'primary_metric_by_model_task.csv', index=False)
display(primary)

primary_rank_order = (
    primary.groupby('model_label', observed=False)['mean_city_rank']
    .mean()
    .sort_values()
    .index
    .tolist()
)
heat = primary.pivot(index='model_label', columns='task', values='mean').reindex(
    index=primary_rank_order, columns=TASK_ORDER
)
rank_heat = primary.pivot(index='model_label', columns='task', values='mean_city_rank').reindex(
    index=primary_rank_order, columns=TASK_ORDER
)
fig, axes = plt.subplots(1, 2, figsize=(22, 5.8))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='vlag', center=0, ax=axes[0])
axes[0].set_title('Primary metric mean by model and task')
axes[0].set_xlabel('')
axes[0].set_ylabel('')
axes[0].set_xticklabels([TASK_TITLE[t] for t in TASK_ORDER], rotation=35, ha='right')
sns.heatmap(rank_heat.astype(float), annot=True, fmt='.2f', cmap='viridis_r', vmin=1, vmax=len(MODEL_ORDER), ax=axes[1])
axes[1].set_title('Mean city rank by task; 1 is best')
axes[1].set_xlabel('')
axes[1].set_ylabel('')
axes[1].set_xticklabels([TASK_TITLE[t] for t in TASK_ORDER], rotation=35, ha='right')
fig.tight_layout()
out = FIG_DIR / 'primary_metric_value_and_rank_heatmaps.png'
fig.savefig(out, dpi=240, bbox_inches='tight')
print(out)
display(fig)
plt.close(fig)



In [ ]:
# Export the released main table from the city-level primary metrics.
main_long = primary[['model', 'model_label', 'task', 'metric', 'mean', 'std', 'n']].copy()
main_long = main_long.rename(columns={'mean': 'avg', 'std': 'c_std', 'n': 'n_cities'})
main_long['model'] = pd.Categorical(main_long['model'], MODEL_ORDER, ordered=True)
main_long['task'] = pd.Categorical(main_long['task'], TASK_ORDER, ordered=True)
main_long = main_long.sort_values(['model', 'task'])
main_long.to_csv(TABLE_DIR / 'main_table_avg_cstd_long.csv', index=False)

wide_rows = []
for model in MODEL_ORDER:
    row = {'model': model, 'model_label': MODEL_LABEL[model]}
    for task in TASK_ORDER:
        part = main_long[(main_long['model'].astype(str) == model) & (main_long['task'].astype(str) == task)]
        prefix = TASK_TITLE[task]
        metric = PRIMARY_METRIC[task]
        if part.empty:
            row[f'{prefix}_Avg_{metric}'] = np.nan
            row[f'{prefix}_CStd'] = np.nan
            row[f'{prefix}_n_cities'] = 0
        else:
            row[f'{prefix}_Avg_{metric}'] = float(part['avg'].iloc[0])
            row[f'{prefix}_CStd'] = float(part['c_std'].iloc[0])
            row[f'{prefix}_n_cities'] = int(part['n_cities'].iloc[0])
    wide_rows.append(row)
main_wide = pd.DataFrame(wide_rows)
main_wide.to_csv(TABLE_DIR / 'main_table_avg_cstd_wide.csv', index=False)

paper = pd.DataFrame({'model': main_wide['model'], 'model_label': main_wide['model_label']})
paper_with_n = pd.DataFrame({'model_label': main_wide['model_label']})
for task in TASK_ORDER:
    prefix = TASK_TITLE[task]
    metric = PRIMARY_METRIC[task]
    avg_col = f'{prefix}_Avg_{metric}'
    cstd_col = f'{prefix}_CStd'
    n_col = f'{prefix}_n_cities'
    paper[f'{prefix} Avg ({metric})'] = main_wide[avg_col].round(3)
    paper[f'{prefix} C-Std.'] = main_wide[cstd_col].round(3)
    paper_with_n[f'{prefix} Avg ({metric})'] = main_wide[avg_col].round(3)
    paper_with_n[f'{prefix} C-Std.'] = main_wide[cstd_col].round(3)
    paper_with_n[f'{prefix} n'] = main_wide[n_col].astype(int)

# Force the displayed/exported main table to use the paper's requested weak-to-strong order.
_export_order = MODEL_ORDER
paper['model'] = pd.Categorical(paper['model'].astype(str), _export_order, ordered=True)
paper = paper.sort_values('model').reset_index(drop=True)
paper['model'] = paper['model'].astype(str)
paper_with_n['model_label'] = pd.Categorical(
    paper_with_n['model_label'].astype(str), [MODEL_LABEL[m] for m in _export_order], ordered=True
)
paper_with_n = paper_with_n.sort_values('model_label').reset_index(drop=True)
paper_with_n['model_label'] = paper_with_n['model_label'].astype(str)
paper.to_csv(RESULT_DIR / 'main_table_avg_cstd_paper.csv', index=False)
paper_with_n.to_csv(TABLE_DIR / 'main_table_avg_cstd_paper_with_n.csv', index=False)
display(paper)
print(RESULT_DIR / 'main_table_avg_cstd_paper.csv')
print(TABLE_DIR / 'main_table_avg_cstd_long.csv')

In [ ]:
mean_city_rank_summary = primary.groupby(['model','model_label']).agg(
    mean_city_rank=('mean_city_rank','mean'),
    median_mean_city_rank=('mean_city_rank','median'),
    best_mean_city_rank=('mean_city_rank','min'),
    worst_mean_city_rank=('mean_city_rank','max'),
    task_count=('task','count'),
).reset_index()

mean_city_rank_summary['run_count'] = mean_city_rank_summary['model'].map(
    coverage.sum(axis=1).to_dict()
).astype(int)
mean_city_rank_summary['coverage_ratio'] = (
    mean_city_rank_summary['run_count'] / (len(CITY_ORDER) * len(TASK_ORDER))
)

# This is a ranking diagnostic, so display/export it in rank order.
mean_city_rank_summary = mean_city_rank_summary.sort_values(
    ['mean_city_rank', 'coverage_ratio'],
    ascending=[True, False],
).reset_index(drop=True)

mean_city_rank_summary.to_csv(DIAG_DIR / 'overall_mean_city_rank.csv', index=False)
display(mean_city_rank_summary)

mean_city_rank_order = mean_city_rank_summary['model_label'].astype(str).tolist()
mean_city_rank_heat = primary.pivot(
    index='model_label',
    columns='task',
    values='mean_city_rank',
).reindex(
    index=mean_city_rank_order,
    columns=TASK_ORDER,
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5.2))

sns.heatmap(
    mean_city_rank_heat.astype(float),
    annot=True,
    fmt='.2f',
    cmap='viridis_r',
    vmin=1,
    vmax=len(MODEL_ORDER),
    ax=axes[0],
)
axes[0].set_title('Mean city rank by task; 1 is best')
axes[0].set_xlabel('')
axes[0].set_ylabel('')
axes[0].set_xticklabels([TASK_TITLE[t] for t in TASK_ORDER], rotation=35, ha='right')

sns.barplot(
    data=mean_city_rank_summary,
    x='mean_city_rank',
    y='model_label',
    hue='model_label',
    order=mean_city_rank_order,
    palette=PALETTE,
    legend=False,
    ax=axes[1],
)
axes[1].invert_xaxis()
axes[1].set_title('Mean city rank across available tasks')
axes[1].set_xlabel('Mean city rank; lower is better')
axes[1].set_ylabel('')

fig.tight_layout()
out = FIG_DIR / 'task_mean_city_rank_heatmap_and_average_rank.png'
fig.savefig(out, dpi=240, bbox_inches='tight')
print(out)
display(fig)
plt.close(fig)



## Representative GDP and LST Metric Bars


In [ ]:
CITY_LABEL = {
    'london':'London', 'new_york':'New York', 'singapore':'Singapore', 'sydney':'Sydney',
    'mumbai':'Mumbai', 'nairobi':'Nairobi', 'jakarta':'Jakarta', 'cape_town':'Cape Town',
}

EXAMPLE_TASKS = [('gdp', 'GDP'), ('lst_day_mean', 'LST')]
EXAMPLE_METRICS = [
    ('R2', '$R^2 \\uparrow$'),
    ('MAE', 'MAE $\\downarrow$'),
    ('RMSE', 'RMSE $\\downarrow$'),
]

def _example_task_data(task):
    data = summary[
        (summary['task'].astype(str).eq(task)) &
        (summary['model'].astype(str).isin(MODEL_ORDER))
    ].copy()
    data['model_str'] = data['model'].astype(str)
    data['city_str'] = data['city'].astype(str)
    data['model_label'] = data['model_str'].map(MODEL_LABEL)
    data['city_label'] = data['city_str'].map(CITY_LABEL)
    data['model_label'] = pd.Categorical(
        data['model_label'], [MODEL_LABEL[m] for m in MODEL_ORDER], ordered=True
    )
    data['city_label'] = pd.Categorical(
        data['city_label'], [CITY_LABEL[c] for c in CITY_ORDER], ordered=True
    )
    return data.sort_values(['city_label', 'model_label'])

fig, axes = plt.subplots(
    len(EXAMPLE_TASKS), len(EXAMPLE_METRICS),
    figsize=(17.5, 7.2),
    sharex=False,
    sharey=False,
)

handles = labels = None
city_labels = [CITY_LABEL[c] for c in CITY_ORDER]
model_labels = [MODEL_LABEL[m] for m in MODEL_ORDER]
for row, (task, task_title) in enumerate(EXAMPLE_TASKS):
    data = _example_task_data(task)
    for col, (metric, metric_label) in enumerate(EXAMPLE_METRICS):
        ax = axes[row, col]
        sub = data.dropna(subset=[metric]).copy()
        handles, labels = _plot_grouped_bars_with_std(
            ax, sub,
            x_col='city_label', y_col=metric, std_col=None,
            hue_col='model_label', x_order=city_labels, hue_order=model_labels,
            palette=PALETTE, capsize=0.0, lw=0.0,
        )
        if metric == 'R2':
            ax.axhline(0, color='black', linewidth=0.8)
            ax.set_ylim(-0.05, 0.88)

        ax.set_title(f'{task_title}: {metric_label}', fontsize=12)
        ax.set_xlabel('')
        ax.set_ylabel(metric_label)
        if row == 0:
            ax.tick_params(axis='x', labelbottom=False, length=0)
        else:
            ax.tick_params(axis='x', rotation=28, labelsize=8.5)

fig.legend(
    handles, labels,
    title=None,
    loc='lower center',
    ncol=6,
    frameon=False,
    bbox_to_anchor=(0.5, 0.015),
    columnspacing=1.25,
    handlelength=1.55,
    handletextpad=0.45,
)
fig.subplots_adjust(left=0.055, right=0.995, top=0.93, bottom=0.20, hspace=0.35, wspace=0.24)

out = FIG_DIR / 'example_gdp_lst_r2_mae_rmse_bars.png'
fig.savefig(out, dpi=260)
print(out)
display(fig)
plt.close(fig)

plot_values = pd.concat([
    _example_task_data(task).assign(task_for_plot=task, task_title=title)
    for task, title in EXAMPLE_TASKS
], ignore_index=True)

plot_values[
    ['task_for_plot','task_title','city_str','city_label','model_str','model_label',
     'R2','MAE','RMSE','R2_std','MAE_std','RMSE_std','n_seeds','n_samples']
].to_csv(
    TABLE_DIR / 'example_gdp_lst_r2_mae_rmse_city_model_values.csv',
    index=False,
)

display(plot_values[['task_for_plot','city_label','model_label','R2','MAE','RMSE']].head())



## Mean City Rank Summary


In [ ]:
mean_city_rank_summary = primary.groupby(['model','model_label']).agg(
    mean_city_rank=('mean_city_rank','mean'),
    median_mean_city_rank=('mean_city_rank','median'),
    best_mean_city_rank=('mean_city_rank','min'),
    worst_mean_city_rank=('mean_city_rank','max'),
    task_count=('task','count'),
).reset_index()

mean_city_rank_summary['run_count'] = mean_city_rank_summary['model'].map(
    coverage.sum(axis=1).to_dict()
).astype(int)
mean_city_rank_summary['coverage_ratio'] = (
    mean_city_rank_summary['run_count'] / (len(CITY_ORDER) * len(TASK_ORDER))
)

# This is a ranking diagnostic, so display/export it in rank order.
mean_city_rank_summary = mean_city_rank_summary.sort_values(
    ['mean_city_rank', 'coverage_ratio'],
    ascending=[True, False],
).reset_index(drop=True)

mean_city_rank_summary.to_csv(DIAG_DIR / 'overall_mean_city_rank.csv', index=False)
display(mean_city_rank_summary)

mean_city_rank_order = mean_city_rank_summary['model_label'].astype(str).tolist()
mean_city_rank_heat = primary.pivot(
    index='model_label',
    columns='task',
    values='mean_city_rank',
).reindex(
    index=mean_city_rank_order,
    columns=TASK_ORDER,
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5.2))

sns.heatmap(
    mean_city_rank_heat.astype(float),
    annot=True,
    fmt='.2f',
    cmap='viridis_r',
    vmin=1,
    vmax=len(MODEL_ORDER),
    ax=axes[0],
)
axes[0].set_title('Mean city rank by task; 1 is best')
axes[0].set_xlabel('')
axes[0].set_ylabel('')
axes[0].set_xticklabels([TASK_TITLE[t] for t in TASK_ORDER], rotation=35, ha='right')

sns.barplot(
    data=mean_city_rank_summary,
    x='mean_city_rank',
    y='model_label',
    hue='model_label',
    order=mean_city_rank_order,
    palette=PALETTE,
    legend=False,
    ax=axes[1],
)
axes[1].invert_xaxis()
axes[1].set_title('Mean city rank across available tasks')
axes[1].set_xlabel('Mean city rank; lower is better')
axes[1].set_ylabel('')

fig.tight_layout()
out = FIG_DIR / 'task_mean_city_rank_heatmap_and_average_rank.png'
fig.savefig(out, dpi=240, bbox_inches='tight')
print(out)
display(fig)
plt.close(fig)


## Complete Per-Task Bar Figures

In [ ]:
def _metric_std_column(metric):
    std_col = f'{metric}_std'
    if std_col in summary.columns:
        return std_col
    if metric == 'F1' and 'F1_macro_std' in summary.columns:
        return 'F1_macro_std'
    return None

def plot_task_metrics(task, metrics):
    data = summary[summary['task'].astype(str) == task].copy()
    fig, axes = plt.subplots(1, len(metrics), figsize=(6.2 * len(metrics) + 3, 5.0), sharex=True)
    if len(metrics) == 1:
        axes = [axes]
    city_order = CITY_ORDER
    model_order = [MODEL_LABEL[m] for m in MODEL_ORDER]
    handles = labels = None
    for ax, metric in zip(axes, metrics):
        sub = data.dropna(subset=[metric]).copy()
        sub['city'] = pd.Categorical(sub['city'].astype(str), city_order, ordered=True)
        sub['model_label'] = pd.Categorical(sub['model_label'].astype(str), model_order, ordered=True)
        sub = sub.sort_values(['city', 'model_label'])
        handles, labels = _plot_grouped_bars_with_std(
            ax, sub,
            x_col='city', y_col=metric, std_col=_metric_std_column(metric),
            hue_col='model_label', x_order=city_order, hue_order=model_order,
            palette=PALETTE, capsize=1.6, lw=0.5,
        )
        ax.set_title(f'{TASK_TITLE[task]} {metric}')
        ax.set_xlabel('City')
        ax.set_ylabel(metric)
        ax.tick_params(axis='x', rotation=25)
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()
    axes[-1].legend(handles, labels, title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout()
    out = FIG_DIR / f'task_{task}_{"_".join(metrics)}.png'
    fig.savefig(out, dpi=200, bbox_inches='tight')
    print(out)
    return fig

def plot_task_primary_city_facets(task):
    metric = PRIMARY_METRIC[task]
    data = summary[summary['task'].astype(str) == task].dropna(subset=[metric]).copy()
    fig, axes = plt.subplots(2, 4, figsize=(22, 8.2), sharey=False)
    axes = axes.ravel()
    model_order = [MODEL_LABEL[m] for m in MODEL_ORDER]
    for ax, city in zip(axes, CITY_ORDER):
        sub = data[data['city'].astype(str) == city].copy()
        sub['model_label'] = pd.Categorical(sub['model_label'].astype(str), model_order, ordered=True)
        sub = sub.sort_values('model_label')
        _plot_grouped_bars_with_std(
            ax, sub,
            x_col='model_label', y_col=metric, std_col=_metric_std_column(metric),
            hue_col=None, x_order=model_order, palette=PALETTE,
            capsize=1.8, lw=0.55, group_width=0.82,
        )
        ax.set_title(city)
        ax.set_xlabel('')
        ax.set_ylabel(metric)
        ax.tick_params(axis='x', rotation=70, labelsize=8)
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()
    fig.suptitle(f'{TASK_TITLE[task]} primary metric: {metric}', y=1.02, fontsize=16)
    fig.tight_layout()
    out = FIG_DIR / f'city_facets_{task}_{metric}.png'
    fig.savefig(out, dpi=200, bbox_inches='tight')
    print(out)
    return fig

for task in TASK_ORDER:
    fig = plot_task_metrics(task, TASK_METRICS[task])
    display(fig)
    plt.close(fig)
    fig = plot_task_primary_city_facets(task)
    display(fig)
    plt.close(fig)

